<a href="https://colab.research.google.com/github/AleynaCifcibasi/Python-ile-Machine-Learning-al-malar-ve-projeleri/blob/main/Python_Oturum_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#RNN


##Tokenizer

In [1]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
veri = [
    'Bu bir örnek cümle.',
    'Başka bir cümle de var.',
    'Farklı bir örnek metin.'
]

In [3]:
tokenizer = Tokenizer() # oov_token="<OOV>"
tokenizer.fit_on_texts(veri)
sayisal_karsilik = tokenizer.texts_to_sequences(veri)

In [4]:
print("Kelime indeksleri : ",tokenizer.word_index)

Kelime indeksleri :  {'bir': 1, 'örnek': 2, 'cümle': 3, 'bu': 4, 'başka': 5, 'de': 6, 'var': 7, 'farklı': 8, 'metin': 9}


In [5]:
for sayi in sayisal_karsilik:
  print(sayi)

[4, 1, 2, 3]
[5, 1, 3, 6, 7]
[8, 1, 2, 9]


In [6]:
sabit_uzunluk = pad_sequences(sayisal_karsilik,maxlen=5)

In [7]:
for sayi in sabit_uzunluk:
  print(sayi)

[0 4 1 2 3]
[5 1 3 6 7]
[0 8 1 2 9]


##Embedding

In [8]:
from keras.layers import Embedding
import numpy as np

In [9]:
giris_boyutu = len(tokenizer.word_index) + 1

boyut = 300

embedding_layer = Embedding(
    input_dim=giris_boyutu,
    output_dim=boyut,
    input_length=5
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [10]:
kimlikli_veri = embedding_layer(np.array(sabit_uzunluk))

In [11]:
print("Embedding Katman Sonucu : ")
print(kimlikli_veri)

Embedding Katman Sonucu : 
tf.Tensor(
[[[ 0.04894736  0.02454361 -0.02026284 ... -0.02735173  0.02513656
   -0.02506275]
  [ 0.03365158 -0.03786968 -0.02460266 ...  0.01986854  0.01104568
    0.03588191]
  [-0.03713952  0.01185962  0.02295193 ... -0.03230015  0.01376339
    0.0363108 ]
  [-0.02999607  0.0279761  -0.04382641 ...  0.00267916  0.02739168
    0.02890911]
  [-0.0113835   0.03710449 -0.0289479  ...  0.02404647 -0.00196836
   -0.03675021]]

 [[-0.02375419 -0.00277953 -0.04152794 ...  0.04248042  0.02583935
   -0.0352548 ]
  [-0.03713952  0.01185962  0.02295193 ... -0.03230015  0.01376339
    0.0363108 ]
  [-0.0113835   0.03710449 -0.0289479  ...  0.02404647 -0.00196836
   -0.03675021]
  [-0.04087826  0.03136135 -0.01435918 ...  0.00496198  0.00507297
    0.02605123]
  [-0.013982   -0.01915703 -0.02315121 ... -0.02373534  0.03520625
    0.01367028]]

 [[ 0.04894736  0.02454361 -0.02026284 ... -0.02735173  0.02513656
   -0.02506275]
  [ 0.04042718  0.0316732  -0.04529816 ...  0

##RNN

In [12]:
!pip install joblib
!pip install colorama

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from keras.layers import SimpleRNN, Dense
from keras.utils import to_categorical
import joblib
from colorama import Fore, Back, Style
from google.colab import drive

In [14]:
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
veri_yolu = '/content/drive/MyDrive/Tech İstanbul/phishing_email.csv'
model_yolu = '/content/drive/MyDrive/Tech İstanbul/rnn.h5'

In [23]:
mail_verisi = pd.read_csv(veri_yolu)
mail_verisi = mail_verisi.where((pd.notnull(mail_verisi)),'')

In [25]:
print("Veri boyutu:", mail_verisi)
print(mail_verisi.head())

Veri boyutu:                                            text_combined  label
0      hpl nom may 25 2001 see attached file hplno 52...      0
1      nom actual vols 24 th forwarded sabrae zajac h...      0
2      enron actuals march 30 april 1 201 estimated a...      0
3      hpl nom may 30 2001 see attached file hplno 53...      0
4      hpl nom june 1 2001 see attached file hplno 60...      0
...                                                  ...    ...
82481  info advantageapartmentscom infoadvantageapart...      1
82482  monkeyorg helpdeskmonkeyorg monkeyorg hi josep...      1
82483  help center infohelpcentercoza_infohelpcenterc...      1
82484  metamask infosofamekarcom verify metamask wall...      1
82485  fastway infofastwaycoza_infofastwaycoza_infofa...      1

[82486 rows x 2 columns]
                                       text_combined  label
0  hpl nom may 25 2001 see attached file hplno 52...      0
1  nom actual vols 24 th forwarded sabrae zajac h...      0
2  enron actu

In [29]:
X = mail_verisi['text_combined']
y = mail_verisi['label']

In [30]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X)
indeksli_veri = tokenizer.texts_to_sequences(X)

In [31]:
word_index = tokenizer.word_index
kelime_uzunlugu = 1000
sabit_uzunluk = pad_sequences(indeksli_veri, maxlen=kelime_uzunlugu)

In [32]:
y = to_categorical(y)

In [33]:
X_train, X_test, y_train, y_test = train_test_split(sabit_uzunluk,y, test_size=0.2, random_state=42)

In [34]:
model = Sequential()
model.add(Embedding(len(word_index)+1, output_dim=128, input_length=kelime_uzunlugu))
model.add(SimpleRNN(64,activation='relu'))
model.add(Dense(2,activation='softmax'))
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [35]:
model.fit(X_train,y_train, epochs=5, batch_size=32, validation_data=(X_test,y_test))

Epoch 1/5
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 200s 94ms/step - accuracy: 0.9443 - loss: 2.6701 - val_accuracy: 0.9874 - val_loss: 0.0376
Epoch 2/5
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 189s 92ms/step - accuracy: 0.9968 - loss: 0.0093 - val_accuracy: 0.9899 - val_loss: 0.0347
Epoch 3/5
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 189s 92ms/step - accuracy: 0.9955 - loss: 0.0248 - val_accuracy: 0.9857 - val_loss: 0.0445
Epoch 4/5
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 190s 92ms/step - accuracy: 0.9996 - loss: 0.0016 - val_accuracy: 0.9867 - val_loss: 0.0432
Epoch 5/5
2063/2063 ━━━━━━━━━━━━━━━━━━━━ 190s 92ms/step - accuracy: 0.9880 - loss: 0.0443 - val_accuracy: 0.9733 - val_loss: 0.0748


In [36]:
model.save(model_yolu)

In [37]:
import tensorflow as tf
model = tf.keras.models.load_model(model_yolu, compile=False)
#Eğitim yapılacağı zaman compile parametresi kaldır ve aşağıdakini çalıştır.
#model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [38]:
while True:
  giris = input('Email mesajını giriniz : ')
  sayisal_kelime = tokenizer.texts_to_sequences([giris])
  sabit_uzunluk = pad_sequences(sayisal_kelime, maxlen=kelime_uzunlugu)
  tahmin = model.predict(sabit_uzunluk)
  if np.argmax(tahmin) == 1:
    print(Fore.RED, 'Bu bir kimlik avı emaili olabilir.')
  else:
    print(Fore.GREEN, 'Bu bir güvenli emaildir.')
  print(Fore.BLUE, 'Doğruluk puanı : ', tahmin[0][np.argmax(tahmin)])

Email mesajını giriniz : Merhaba
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
 Bu bir kimlik avı emaili olabilir.
 Doğruluk puanı :  0.7107077
Email mesajını giriniz : hpl nom may 25 2001 see attached file hplno 525 xls hplno 525 xls
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
 Bu bir güvenli emaildir.
 Doğruluk puanı :  0.9999999


KeyboardInterrupt: Interrupted by user